In [ ]:
import io
from datetime import datetime
from operator import itemgetter
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import requests

In [ ]:
UCI_DATA_URL = "https://archive.ics.uci.edu/static/public/321/electricityloaddiagrams20112014.zip"

FREQUENCY_MINUTES = 15

### UCI Electricity Load Dataset

In [ ]:
# Download data

def load_uci_data(
    target_dir: str,
    target_file: str | None = "LD2011_2014.txt",
    url: str = UCI_DATA_URL
):
    response = requests.get(UCI_DATA_URL)
    try: 
        response.raise_for_status()
    except requests.HTTPError as e:
        print(f"Failed to download from {url}. Reason: {e}")
        return 

    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        try: 
            if target_file:
                z.extract(member=target_file, path=target_dir)
            else: 
                z.extractall(path=target_dir)
        except Exception as e:
            print(f"Failed to extract files. Reason: {e}")
            return


In [ ]:
# Load data as polars dataframe
UCI_DF = pl.read_csv(
    "./data/LD2011_2014.txt",
    has_header=True,
    separator=";",
    decimal_comma=True,
    try_parse_dates=True,
    infer_schema_length=1_000_000
)

# First column should be timestamp column
UCI_DF = UCI_DF.rename({UCI_DF.columns[0]: "timestamp"}).sort(by="timestamp")

### EDA

In [ ]:
# Duplicate timestamps

duplicate_timestamps = UCI_DF.select(pl.col("timestamp")).is_duplicated().sum()
print(f"Number of duplicate timestamps found: {duplicate_timestamps}")

In [ ]:
# Timestamp for every interval

min_timestamp = UCI_DF.get_column("timestamp").min()
max_timestamp = UCI_DF.get_column("timestamp").max()

expected_timestamps = pl.datetime_range(
    start=min_timestamp,
    end=max_timestamp,
    interval=f"{FREQUENCY_MINUTES}m",
    closed="both",
    eager=True,
)

print("Expected number of timestamps: ", len(expected_timestamps))
print("Actual number of timestamps: ", len(UCI_DF))

In [ ]:
# Number of non-zero observations

non_zero_counts = (UCI_DF.select(pl.exclude("timestamp")) > 0).sum()

plt.hist(non_zero_counts.to_numpy().flatten(), bins=20, color="tab:blue", alpha=0.75)
plt.axvline(non_zero_counts.to_numpy().flatten().min(), color="grey", ls="--")
plt.axvline(non_zero_counts.to_numpy().flatten().max(), color="grey", ls="--")
plt.xlabel("Non-zero count")
plt.ylabel("Frequency");

In [ ]:
# First and last observations for each client

def get_min_max_timestamps_by_client(uci_df: pl.DataFrame) -> pl.DataFrame:
    min_max_ts = (
        uci_df.unpivot(
            on=[c for c in UCI_DF.columns if c != "timestamp"],
            index="timestamp",
            variable_name="client"
        )
        .filter(pl.col("value") > 0)
        .group_by("client", maintain_order=True)
        .agg(
            min_timestamp=pl.col("timestamp").min(),
            max_timestamp=pl.col("timestamp").max()
        )
    )
    return min_max_ts


min_max_ts_by_client = get_min_max_timestamps_by_client(UCI_DF)

# Plot
first_ts = min_max_ts_by_client["min_timestamp"].to_list()
last_ts = min_max_ts_by_client["max_timestamp"].to_list()

fig, ax = plt.subplots(1, 1)
ax.scatter(first_ts, np.arange(len(first_ts)), alpha=0.1, color="tab:blue", label="first")
ax.scatter(last_ts, np.arange(len(last_ts)), alpha=0.1, color="tab:red", label="last")
ax.legend()
for tick in ax.get_xticklabels():
    tick.set_rotation(45)
ax.set(xlabel="Timestamp", ylabel="Client")
fig.align_labels()
fig.tight_layout();

In [ ]:
# Are timestamps continuous?
def get_non_continuous_timeseries_by_client(uci_df: pl.DataFrame) -> dict[str, pl.DataFrame]:
    non_cont_ts: dict[str, pl.DataFrame] = {}

    for col in uci_df.columns:
        if col == "timestamp":
            continue
        
        result = (
            # select timestamp and col columns
            uci_df.select(pl.col("timestamp"), pl.col(col))
            # filter for where col > 0, sort by timestamp
            .filter(pl.col(col) > 0).sort(by="timestamp")
            # add column for first diff on timestamp
            .with_columns(pl.col("timestamp").diff().alias("timestamp_diff"))
            # filer result to where timestamp_diff > FREQUENCY
            .filter(pl.col("timestamp_diff").dt.total_seconds() > FREQUENCY_MINUTES * 60)
        )

        if result.is_empty():
            continue

        non_cont_ts[col] = result
    
    return non_cont_ts

non_continuous_timeseries = get_non_continuous_timeseries_by_client(UCI_DF)


In [ ]:
# Get proportion of non-continuous timestamps as a function of total
# length of timeseries
pct_non_continous: dict[str, tuple[float, float]] = {}
for client, non_cont_client_df in non_continuous_timeseries.items():
    # Get min / max timestamps for this client
    client_min_max_ts = min_max_ts_by_client.filter(pl.col("client") == client)
    [client_min_ts] = client_min_max_ts["min_timestamp"].to_list()
    [client_max_ts] = client_min_max_ts["max_timestamp"].to_list()
    
    expected_client_ts = pl.datetime_range(
        start=client_min_ts,
        end=client_max_ts,
        interval=f"{FREQUENCY_MINUTES}m",
        closed="both",
        eager=True,   
    )
    n_expected_client_ts = len(expected_client_ts)

    # Calculate number of non-continuous timestamps
    [n_non_cont_client_ts] = (
        non_cont_client_df
        # Calculate number of freq multiples of timestamp diff
        # Subtract 1.0 since a multiple of 1.0 corresponds to "cont" timestamp diff
        .select(freq_multiple=(pl.col("timestamp_diff") / pl.duration(minutes=FREQUENCY_MINUTES)) - 1.0)
        # Sum across all timestamps
        .sum()
        ["freq_multiple"]
        .to_list()
    )
    
    pct_non_continous[client] = (n_non_cont_client_ts, n_expected_client_ts)

In [ ]:
# Plot
non_cont_clients_and_pct = [
    (client, (n / d))
    for (client, (n, d)) in pct_non_continous.items()
]
non_cont_clients_and_pct = sorted(non_cont_clients_and_pct, key=itemgetter(1))

non_cont_client_names, non_cont_client_pcts = zip(*non_cont_clients_and_pct)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

# Distribution of proportion of non cont timestamps
axes[0].hist(non_cont_client_pcts, bins=50, alpha=0.5);
axes[0].set(xlabel="Fraction Non Continuous Timestamps", ylabel="Counts")

# Scatter plot by client id for pct > 1%
large_non_cont_pct = [(c, pct) for (c, pct) in non_cont_clients_and_pct if pct > 0.005]
large_non_cont_client_names, large_non_cont_client_pcts = zip(*large_non_cont_pct)
axes[1].scatter(large_non_cont_client_names, large_non_cont_client_pcts, alpha=0.5);
axes[1].set(ylabel="Fraction Non Continuous Timestamps")
for tick in axes[1].get_xticklabels():
    tick.set_rotation(90)

fig.tight_layout();

In [ ]:
# Plot timeseries for clients with large pct of non-continuous timestamsps
idx = 13
client_name = large_non_cont_client_names[idx]
client_pct = large_non_cont_client_pcts[idx]

print(client_name)

# Get min / max timestamps for this client
client_min_max_ts = min_max_ts_by_client.filter(pl.col("client") == client_name)
[client_min_ts] = client_min_max_ts["min_timestamp"].to_list()
[client_max_ts] = client_min_max_ts["max_timestamp"].to_list()

client_df = (
    UCI_DF.select(pl.col("timestamp"), pl.col(client_name))
    .filter(pl.col("timestamp") >= client_min_ts, pl.col("timestamp") <= client_max_ts)
)

fig, ax = plt.subplots(1, 1, figsize=(15, 3.5))

start, stop = int(1 * 10_000), int(2 * 10_000)
ax.plot(
    client_df["timestamp"].to_list()[start:stop],
    client_df[client_name].to_list()[start:stop],
    label=f"{client_name} ({client_pct * 100:.2f} %)",
)
ax.legend()

fig.tight_layout();

In [ ]:
# Data processing

clients_non_cont_but_no_interpolation = ["MT_001", "MT_133", "MT_131", "MT_132", "MT_130", "MT_348", "MT_347"]

clients_to_drop_to_too_many_zeros = ["MT_288", "MT_066", "MT_127", ]

clients_to_slice = ["MT_015"]
to_keep_filters = {"MT_015": (datetime(2013, 11, 19), datetime(2015, 1, 1))}

# For all other clients, interpolation should be fine.
clients_to_interpolate = [
    c 
    for c in non_cont_client_names 
    if c not in sum([clients_non_cont_but_no_interpolation, clients_to_drop_to_too_many_zeros, clients_to_slice], [])
]

In [ ]:
def drop_client_timeseries(uci_df: pl.DataFrame, client_name: str) -> pl.DataFrame:
    return uci_df.select(pl.all().exclude(client_name))


def filter_client_timeseries(
    uci_df: pl.DataFrame,
    client_name: str,
    start_ts: datetime,
    end_ts: datetime,
) -> pl.DataFrame:

    
    client_df = (
        uci_df
        .select(pl.col("timestamp"), pl.col(client_name))
        .filter(pl.col("timestamp").is_between(start_ts, end_ts, closed="both"))
    )

    # Merge back onto main df
    uci_df = (
        uci_df
        .select(pl.all().exclude(client_name))
        .join(client_df, on="timestamp", how="left")
        .select(
            pl.all().exclude(client_name),
            pl.col(client_name).fill_null(0.0)
        )
    )

    # Sort columns
    uci_df = uci_df.select(
        pl.col("timestamp"),
        *[pl.col(c) for c in sorted([c for c in uci_df.columns if c != "timestamp"])]
    )

    return uci_df


def interpolate_client_timeseries(
    uci_df: pl.DataFrame,
    client_name: str,
    start_ts: datetime,
    end_ts: datetime,
) -> pl.DataFrame:
    """
    Filter a client timeseries to between start_ts and end_ts, and interpolate 
    """

    # Get all non-zero observations between start_ts and end_ts
    client_df = (
        uci_df.select(pl.col("timestamp"), pl.col(client_name))
        .filter(
            pl.col("timestamp").is_between(start_ts, end_ts, closed="both"),
            pl.col(client_name) > 0
        )
    )

    # Construct timeseries of expected timestamps between start_ts and end_ts
    expected_ts = pl.datetime_range(
        start=start_ts,
        end=end_ts,
        interval=f"{FREQUENCY_MINUTES}m",
        closed="both",
        eager=True,
    )

    # Merge and interpoalte
    client_df = (
        expected_ts
        .to_frame(name="timestamp")
        .join(client_df, on="timestamp", how="left")
        .select(pl.col("timestamp"), pl.col(client_name).interpolate())
    )

    # Merge back onto original uci_df
    uci_df = (
        uci_df
        .select(pl.all().exclude(client_name))
        .join(client_df, on="timestamp", how="left")
        .select(
            pl.all().exclude(client_name),
            pl.col(client_name).fill_null(0.0)
        )
    )

    # Sort columns
    uci_df = uci_df.select(
        pl.col("timestamp"),
        *[pl.col(c) for c in sorted([c for c in uci_df.columns if c != "timestamp"])]
    )

    return uci_df

In [ ]:
# Drop clients
for client in clients_to_drop_to_too_many_zeros:
    UCI_DF = drop_client_timeseries(uci_df=UCI_DF, client_name=client)

In [ ]:
# Filter client timeseries
for client_name, (start_ts, end_ts) in to_keep_filters.items():
    UCI_DF = filter_client_timeseries(
        uci_df=UCI_DF,
        client_name=client_name,
        start_ts=start_ts,
        end_ts=end_ts
    )

In [ ]:
# Interpolate client timeseries
for client_name in clients_to_interpolate:
    # Get min / max timestamps for this client
    client_min_max_ts = min_max_ts_by_client.filter(pl.col("client") == client_name)
    [client_min_ts] = client_min_max_ts["min_timestamp"].to_list()
    [client_max_ts] = client_min_max_ts["max_timestamp"].to_list()

    UCI_DF = interpolate_client_timeseries(
        uci_df=UCI_DF,
        client_name=client_name,
        start_ts=client_min_ts,
        end_ts=client_max_ts,
    )

In [ ]:
## What do some of the timeseries actually look like?

long_client_demand_table = (
    UCI_DF
    .unpivot(
        on=[c for c in UCI_DF.columns if c != "timestamp"],
        index="timestamp",
        variable_name="client",
        value_name="demand"
    )
    .join(
        other=min_max_ts_by_client,
        on="client",
        how="left",
    )
    .with_columns(
        in_range=(
            pl.col("timestamp")
            .is_between(pl.col("min_timestamp"), pl.col("max_timestamp"))
        ),
        log1p_demand=(
            pl.col("demand").log1p()
        )
    )
)

agg_client_demand = (
    long_client_demand_table
    .group_by(pl.col("timestamp"), pl.col("in_range"))
    .agg(
        demand_mean=pl.col("demand").mean(),
        demand_median=pl.col("demand").median(),
        log1p_demand_mean=pl.col("log1p_demand").mean(),
        log1p_demand_median=pl.col("log1p_demand").median(),
    )
    .filter(pl.col("in_range"))
    .select(pl.all().exclude("in_range"))
    .sort(pl.col("timestamp"))
)

agg_hourly_client_demand = (
    long_client_demand_table
    .filter(pl.col("in_range"))
    .group_by(pl.col("timestamp").dt.truncate(every="1h"))
    .agg(demand_mean=pl.col("demand").mean(), demand_median=pl.col("demand").median())
    .sort(pl.col("timestamp"))
)

In [ ]:
# Timeseries plots

fig, ax = plt.subplots(2, 1, figsize=(15, 6), sharex=True)

start_ts, end_ts = (datetime(2012, 10, 1), datetime(2013, 1, 1))

# --- Consumption data for each individual client ---
long_client_demand_table_filtered = (
    long_client_demand_table
    .filter(
        pl.col("timestamp").is_between(start_ts, end_ts, closed="left"),
        pl.col("in_range")
    )
)
active_clients = long_client_demand_table_filtered["client"].unique().to_list()
for client in active_clients:
    client_df = (
        long_client_demand_table_filtered
        .filter(pl.col("client") == client)
        .select(pl.col("timestamp"), pl.col("demand"), pl.col("log1p_demand"))
    )
    for i, col in enumerate(["demand", "log1p_demand"]):
        ax[i].plot(
            client_df["timestamp"].to_list(),
            client_df[col].to_list(),
            alpha=0.5,
            color="grey",
            lw=0.2
        )

# --- Mean consumption data for active clients ---
agg_client_demand_filtered = (
    agg_client_demand
    .filter(pl.col("timestamp").is_between(start_ts, end_ts, closed="left"))
)
for i, mean_col in enumerate(["demand_mean", "log1p_demand_mean"]):
    ax[i].plot(
        agg_client_demand_filtered["timestamp"].to_list(),
        agg_client_demand_filtered[mean_col].to_list(),
        lw=2.0,
        label="Median",
        color="tab:blue",
    )

# --- Median consumption data for active clients ---
for i, median_col in enumerate(["demand_median", "log1p_demand_median"]):
    ax[i].plot(
        agg_client_demand_filtered["timestamp"].to_list(),
        agg_client_demand_filtered[median_col].to_list(),
        lw=2.0,
        label="Median",
        color="tab:orange",
    )


for i in range(2):
    ax[i].legend(ncols=2)
    ax[i].set(
        yscale="log" if i == 0 else "linear",
        ylabel="Demand (kW)" if i == 0 else "Log1p Demand (kW)",
    )

fig.tight_layout();

In [ ]:
# Distribution plots

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
start_ts, end_ts = (datetime(2011, 1, 1), datetime(2015, 1, 1))
long_client_demand_table_filtered = (
    long_client_demand_table
    .filter(
        pl.col("timestamp").is_between(start_ts, end_ts, closed="left"),
        pl.col("in_range")
    )
)
for i, demand_col in enumerate(["demand", "log1p_demand"]):
    ax[i].hist(
        long_client_demand_table_filtered[demand_col].to_list(),
        alpha=0.75,
        bins=5000 if i == 0 else 50,
        color="tab:blue",
        density=True,
        lw=0.2
    )
    ax[i].set(
        xscale="log" if i == 0 else "linear",
        xlabel="Demand (kW)" if i == 0 else "Log1p Demand (kW)",
        ylabel="Density" if i == 0 else "",
    )

fig.align_labels()
fig.tight_layout();

In [ ]:
# NEXT QUESTIONS:
# Timeseries clustering?
# - https://arxiv.org/html/2412.20582v1

# Data Transformations
# Log, Box Cox, Standard Scaler ?